<a href="https://colab.research.google.com/github/TuNgoc233/ETL_CaNhan_2/blob/master/Offline_recommender.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Offline Recommender Pipeline - 5-Star Hybrid

Hệ thống gợi ý offline theo flow thống nhất:

1. Setup
2. Load & Merge Metadata
3. Content-Based
4. Collaborative Filtering - Funk SVD
5. Hybrid Union (CB, CF) theo `city_name`
6. Test
7. Đánh giá mô hình bằng RMSE cho Content-Based và Collaborative Filtering
8. Trích xuất dữ liệu mẫu ra CSV

Lưu ý: thành phố được lấy từ cột `City` trong `places_lookup.csv`, không dùng `city_id`.

## 1. Setup

In [ ]:
# === Google Colab setup ===
# 1) Mount Google Drive (chỉ chạy trên Colab)
try:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 2) Cài thư viện Colab chưa có sẵn
if IN_COLAB:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                    'sentence-transformers'], check=True)

import os, json, pickle, time
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.sparse import load_npz
from sentence_transformers import SentenceTransformer

# 3) Đường dẫn — giả định folder `Recommendation_System` nằm trực tiếp trong `MyDrive`
#    (tức là Drive sẽ có: MyDrive/Recommendation_System/places_lookup.csv, ...)
if IN_COLAB:
    BASE = Path('/content/drive/MyDrive/Recommender System')
else:
    BASE = Path('.').resolve()
    fallback_base = BASE.parent / 'Recommendation_System'
    if not (BASE / 'places_lookup.csv').exists() and (fallback_base / 'places_lookup.csv').exists():
        print(f'Không tìm thấy dữ liệu trong {BASE}, chuyển sang BASE fallback: {fallback_base}')
        BASE = fallback_base

POI_DIR = BASE / 'DATN-ETL' / 'Filtered_Data_Foody'
RES_DIR = BASE / 'DATN-ETL' / 'merged_foody_restaurant'
OUT_DIR = BASE / 'recommender_artifacts'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Sanity check — báo lỗi sớm nếu thiếu file
for p in [BASE / 'places_lookup.csv',
          BASE / 'rating_matrix.npz',
          BASE / 'rating_matrix_users.csv',
          BASE / 'rating_matrix_items.csv',
          POI_DIR, RES_DIR]:
    assert p.exists(), f'Không tìm thấy: {p}'

print('BASE:', BASE)
print('OUT :', OUT_DIR)

## 2. Load & Merge Metadata

In [ ]:
REQUIRED_META_COLUMNS = {'ResID', 'PlaceName', 'City', 'Latitude', 'Longitude'}

def _first_value(rec, keys, default=None):
    for key in keys:
        if key in rec and rec.get(key) not in (None, '', 'null'):
            return rec.get(key)
    return default

def _to_float_or_nan(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return np.nan

def _normalize_city_name(value):
    if pd.isna(value):
        return ''
    return str(value).strip()

def _build_places_meta():
    print('Đang load và merge metadata địa điểm, bao gồm city_name, latitude, longitude...')

    places_lookup = pd.read_csv(BASE / 'places_lookup.csv', encoding='utf-8-sig')
    places_lookup.columns = [c.strip() for c in places_lookup.columns]
    places_lookup['ResID'] = places_lookup['ResID'].astype(int)
    places_lookup['City'] = places_lookup['City'].apply(_normalize_city_name)
    places_lookup['PlaceName'] = places_lookup['PlaceName'].fillna('').astype(str)

    poi_meta = {}
    for city_dir in sorted(POI_DIR.iterdir()):
        if not city_dir.is_dir():
            continue
        for jf in city_dir.glob('*.json'):
            try:
                with open(jf, 'r', encoding='utf-8') as f:
                    data = json.load(f)
            except Exception as e:
                print('Bỏ qua file POI lỗi:', jf, e)
                continue
            for rec in data:
                rid = _first_value(rec, ['id', 'Id', 'ResID'])
                if rid is None:
                    continue
                poi_meta[int(rid)] = {
                    'CategoryGroup': _first_value(rec, ['category_group', 'CategoryGroup'], ''),
                    'CategorySub': _first_value(rec, ['category_sub', 'CategorySub'], ''),
                    'Latitude': _to_float_or_nan(_first_value(rec, ['latitude', 'Latitude', 'lat', 'Lat'])),
                    'Longitude': _to_float_or_nan(_first_value(rec, ['longitude', 'Longitude', 'lng', 'Lng', 'long', 'Long'])),
                }

    res_meta = {}
    for jf in sorted(RES_DIR.glob('*.json')):
        try:
            with open(jf, 'r', encoding='utf-8') as f:
                data = json.load(f)
        except Exception as e:
            print('Bỏ qua file nhà hàng lỗi:', jf, e)
            continue
        for rec in data:
            rid = _first_value(rec, ['Id', 'id', 'ResID'])
            if rid is None:
                continue
            res_meta[int(rid)] = {
                'Latitude': _to_float_or_nan(_first_value(rec, ['Latitude', 'latitude', 'lat', 'Lat'])),
                'Longitude': _to_float_or_nan(_first_value(rec, ['Longitude', 'longitude', 'lng', 'Lng', 'long', 'Long'])),
            }

    rows = []
    for _, r in places_lookup.iterrows():
        rid = int(r['ResID'])
        name = r['PlaceName']
        city_name = _normalize_city_name(r['City'])
        if rid in poi_meta:
            meta = poi_meta[rid]
            rows.append((rid, name, city_name, meta['Latitude'], meta['Longitude'],
                         meta['CategoryGroup'], meta['CategorySub'], 'POI'))
        elif rid in res_meta:
            meta = res_meta[rid]
            rows.append((rid, name, city_name, meta['Latitude'], meta['Longitude'],
                         'Nhà hàng', '', 'Res'))
        else:
            rows.append((rid, name, city_name, np.nan, np.nan, '', '', 'Unknown'))

    places_meta = pd.DataFrame(
        rows,
        columns=['ResID', 'PlaceName', 'City', 'Latitude', 'Longitude',
                 'CategoryGroup', 'CategorySub', 'Kind'],
    )
    places_meta = places_meta.drop_duplicates('ResID').reset_index(drop=True)
    for c in ('PlaceName', 'City', 'CategoryGroup', 'CategorySub'):
        places_meta[c] = places_meta[c].fillna('').astype(str).str.strip()
    places_meta['Latitude'] = pd.to_numeric(places_meta['Latitude'], errors='coerce')
    places_meta['Longitude'] = pd.to_numeric(places_meta['Longitude'], errors='coerce')
    places_meta.to_pickle(OUT_DIR / 'places_meta.pkl')
    return places_meta


if (OUT_DIR / 'places_meta.pkl').exists():
    places_meta = pd.read_pickle(OUT_DIR / 'places_meta.pkl')
    if not REQUIRED_META_COLUMNS.issubset(set(places_meta.columns)) and POI_DIR.exists() and RES_DIR.exists():
        print('places_meta.pkl cũ chưa có đủ City/Latitude/Longitude, đang build lại metadata...')
        places_meta = _build_places_meta()
    else:
        print('Đã load places_meta.pkl hiện có.')
else:
    places_meta = _build_places_meta()

for col in REQUIRED_META_COLUMNS:
    if col not in places_meta.columns:
        places_meta[col] = np.nan

places_meta['ResID'] = places_meta['ResID'].astype(int)
places_meta['City'] = places_meta['City'].apply(_normalize_city_name)
places_meta['Latitude'] = pd.to_numeric(places_meta['Latitude'], errors='coerce')
places_meta['Longitude'] = pd.to_numeric(places_meta['Longitude'], errors='coerce')

res2city = dict(zip(places_meta['ResID'].astype(int).values, places_meta['City'].values))
res2lat = dict(zip(places_meta['ResID'].astype(int).values, places_meta['Latitude'].values))
res2lon = dict(zip(places_meta['ResID'].astype(int).values, places_meta['Longitude'].values))
city_arr = places_meta['City'].values

print(f'places_meta: {places_meta.shape}')
print(places_meta['Kind'].value_counts())
print(f'Số thành phố: {places_meta["City"].nunique()}')
print(f'Số địa điểm có tọa độ: {places_meta[["Latitude", "Longitude"]].dropna().shape[0]}')
print(places_meta.head())

## 3. Content-Based — Semantic Embedding

**Quy trình**:
1. Tạo `content_text = "PlaceName - CategoryGroup - CategorySub"`.
2. Encode bằng `paraphrase-multilingual-MiniLM-L12-v2` rồi L2-normalize → cosine = dot product.
3. Hàm tra cứu `get_top_k_items_by_item(item_id, k)` → tính cosine giữa item và các địa điểm cùng thành phố, lấy top-K.
4. Pre-compute lookup table `Item_ID → List[Similar_Item_IDs]` và lưu xuống file.

In [ ]:
res2content_idx = {int(rid): i for i, rid in enumerate(places_meta['ResID'].values)}
city_arr = places_meta['City'].values

def _same_city_candidate_indices(city_name=None):
    city_name = _normalize_city_name(city_name)
    if city_name:
        return np.where(city_arr == city_name)[0]
    return np.arange(len(places_meta))

def get_top_k_items_by_item(item_id: int, city_name=None, k: int = 50) -> list:
    """Trả về top-K item giống item đang xem, lọc theo tên thành phố."""
    item_id = int(item_id)
    idx = res2content_idx.get(item_id)
    if idx is None:
        return []

    target_city = _normalize_city_name(city_name if city_name is not None else res2city.get(item_id))
    cand_idx = _same_city_candidate_indices(city_name=target_city)
    if len(cand_idx) == 0:
        return []

    sim = embeddings[idx] @ embeddings[cand_idx].T
    order = np.argsort(-sim)
    cand_resids = places_meta['ResID'].values[cand_idx][order]
    return cand_resids[cand_resids != item_id][:k].astype(int).tolist()


if (OUT_DIR / 'content_embeddings.npy').exists() and \
   (OUT_DIR / 'cb_lookup.pkl').exists() and \
   (OUT_DIR / 'embedding_model.txt').exists():
    embeddings = np.load(OUT_DIR / 'content_embeddings.npy')
    with open(OUT_DIR / 'cb_lookup.pkl', 'rb') as f:
        cb_lookup = pickle.load(f)
    with open(OUT_DIR / 'embedding_model.txt', 'r', encoding='utf-8') as f:
        EMBED_MODEL_NAME = f.read().strip()
    print(f'Đã load embedding, CB lookup và model: {EMBED_MODEL_NAME}')
else:
    places_meta['content_text'] = places_meta.apply(
        lambda r: f"{r['PlaceName']} - {r['City']} - {r['CategoryGroup']} - {r['CategorySub']}".strip(),
        axis=1,
    )

    EMBED_MODEL_NAME = 'paraphrase-multilingual-MiniLM-L12-v2'
    print(f'Đang load model embedding: {EMBED_MODEL_NAME}...')
    embed_model = SentenceTransformer(EMBED_MODEL_NAME)

    print('Đang encode embedding nội dung...')
    embeddings = embed_model.encode(
        places_meta['content_text'].tolist(),
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    ).astype(np.float32)
    print(f'embeddings: {embeddings.shape}, dtype={embeddings.dtype}')

    TOP_K_CB = 50
    print(f'\nĐang pre-compute CB lookup (k={TOP_K_CB})...')
    cb_lookup = {}
    all_resids = places_meta['ResID'].astype(int).values
    t0 = time.time()
    for i, rid in enumerate(all_resids, 1):
        cb_lookup[int(rid)] = get_top_k_items_by_item(int(rid), k=TOP_K_CB)
        if i % 5000 == 0 or i == len(all_resids):
            print(f'  [{i}/{len(all_resids)}] elapsed={time.time()-t0:.1f}s', flush=True)

    np.save(OUT_DIR / 'content_embeddings.npy', embeddings)
    with open(OUT_DIR / 'cb_lookup.pkl', 'wb') as f:
        pickle.dump(cb_lookup, f)
    with open(OUT_DIR / 'embedding_model.txt', 'w', encoding='utf-8') as f:
        f.write(EMBED_MODEL_NAME)
    print(f'\nCB lookup: {len(cb_lookup)} items, top-{TOP_K_CB} mỗi item -> cb_lookup.pkl')

## 4. Collaborative Filtering - Funk SVD (Surprise)

Phan nay load ma tran rating, chuan hoa rating ve thang 5 sao, split train/validation/test, train Funk SVD va pre-compute lookup CF.

In [ ]:
import math
import random
import time
import pickle

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, load_npz
from surprise import Dataset, Reader, SVD, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.metrics import mean_squared_error

R_raw = load_npz(BASE / 'rating_matrix.npz').astype(np.float32)
R = R_raw.copy().astype(np.float32)

# Chuan hoa rating ve thang 5 sao.
# Neu du lieu goc dang o thang 10 thi chia 2; neu da la 5 sao thi giu nguyen.
rating_max = float(R.data.max()) if R.nnz else 0.0
if rating_max > 5.0:
    R.data = R.data / 2.0
R.data = np.clip(R.data, 0.5, 5.0).astype(np.float32)

cf_users = pd.read_csv(BASE / 'rating_matrix_users.csv', encoding='utf-8-sig')
cf_items = pd.read_csv(BASE / 'rating_matrix_items.csv', encoding='utf-8-sig')
cf_users.columns = [c.strip() for c in cf_users.columns]
cf_items.columns = [c.strip() for c in cf_items.columns]
cf_user_ids = cf_users.iloc[:, 0].astype(int).values
cf_item_ids = cf_items.iloc[:, 0].astype(int).values
R_csr = R.tocsr()

print(f'R: {R.shape}, nnz={R.nnz}, users={len(cf_user_ids)}, items={len(cf_item_ids)}')
print(f'Rating scale after normalization: min={R.data.min():.2f}, max={R.data.max():.2f}')

R_coo = R_csr.tocoo()
df_ratings = pd.DataFrame({
    'u_idx': R_coo.row,
    'i_idx': R_coo.col,
    'rating': R_coo.data,
})

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(df_ratings[['u_idx', 'i_idx', 'rating']], reader)

raw_ratings = list(data.raw_ratings)
random.seed(42)
random.shuffle(raw_ratings)

total = len(raw_ratings)
test_size = int(0.1 * total)
val_size = int(0.1 * total)
train_size = total - test_size - val_size

train_raw = raw_ratings[:train_size]
val_raw = raw_ratings[train_size:train_size + val_size]
test_raw = raw_ratings[train_size + val_size:]

data.raw_ratings = train_raw
train_set = data.build_full_trainset()
val_set = data.construct_testset(val_raw)
test_set = data.construct_testset(test_raw)

print(f'\nTổng số ratings: {total}')
print(f'Train set size: {len(train_raw)}')
print(f'Validation set size: {len(val_raw)}')
print(f'Test set size: {len(test_raw)}')

print('\nĐang xây dựng R_train_csr từ train set để tránh data leakage...')
train_u_indices = [uid for uid, _, _, _ in train_raw]
train_i_indices = [iid for _, iid, _, _ in train_raw]
train_ratings_data = [rating for _, _, rating, _ in train_raw]

num_users, num_items = R_csr.shape
R_train_csr = csr_matrix(
    (train_ratings_data, (train_u_indices, train_i_indices)),
    shape=(num_users, num_items),
    dtype=np.float32,
)
global_mean_5star = float(np.mean(train_ratings_data))

print('\nĐang tìm kiếm siêu tham số tối ưu cho Funk SVD trên train set...')
param_grid = {
    'n_factors': [16, 32],
    'n_epochs': [10, 20],
    'lr_all': [0.002, 0.005],
    'reg_all': [0.05, 0.1, 0.2],
}

gs = GridSearchCV(SVD, param_grid, measures=['rmse'], cv=3, n_jobs=-1)
gs.fit(data)

best_svd_params = gs.best_params['rmse']
print('Bộ siêu tham số tốt nhất:', best_svd_params)

cf_algo = SVD(**best_svd_params)
cf_algo.fit(train_set)

print('Đang đồng bộ User/Item Factors sang ma trận NumPy để nhân nhanh...')
P = np.zeros((num_users, cf_algo.pu.shape[1]), dtype=np.float32)
b_u = np.zeros(num_users, dtype=np.float32)
for raw_u in range(num_users):
    try:
        inner_u = train_set.to_inner_uid(raw_u)
        P[raw_u] = cf_algo.pu[inner_u]
        b_u[raw_u] = cf_algo.bu[inner_u]
    except ValueError:
        pass

Q = np.zeros((num_items, cf_algo.qi.shape[1]), dtype=np.float32)
b_i = np.zeros(num_items, dtype=np.float32)
for raw_i in range(num_items):
    try:
        inner_i = train_set.to_inner_iid(raw_i)
        Q[raw_i] = cf_algo.qi[inner_i]
        b_i[raw_i] = cf_algo.bi[inner_i]
    except ValueError:
        pass

global_mean = train_set.global_mean



user_id_to_cf_idx = {int(uid): i for i, uid in enumerate(cf_user_ids)}
cf_item_id_to_idx = {int(rid): i for i, rid in enumerate(cf_item_ids)}
cf_item_city_names = np.array([_normalize_city_name(res2city.get(int(rid), '')) for rid in cf_item_ids], dtype=object)

def _city_mask_for_cf(city_name):
    city_name = _normalize_city_name(city_name)
    if not city_name:
        return np.ones(len(cf_item_ids), dtype=bool)
    return cf_item_city_names == city_name

def get_top_k_items_for_user_svd(user_id: int, city_name=None, k: int = 50) -> list:
    """CF Funk SVD: truyền UserID + city_name, trả về top-K địa điểm user có thể thích trong thành phố đó."""
    uid = int(user_id)
    u_idx = user_id_to_cf_idx.get(uid)
    if u_idx is None:
        return []

    preds = global_mean + b_u[u_idx] + b_i + Q.dot(P[u_idx])
    preds = preds.astype(np.float32, copy=True)

    start, end = R_csr.indptr[u_idx], R_csr.indptr[u_idx + 1]
    history_items = R_csr.indices[start:end]
    preds[history_items] = -np.inf

    city_mask = _city_mask_for_cf(city_name)
    preds[~city_mask] = -np.inf

    valid_count = int(np.isfinite(preds).sum())
    if valid_count == 0:
        return []

    k_eff = min(int(k), valid_count)
    if k_eff >= len(preds):
        order = np.argsort(-preds)
    else:
        part = np.argpartition(-preds, kth=k_eff - 1)[:k_eff]
        order = part[np.argsort(-preds[part])]

    order = order[np.isfinite(preds[order])]
    return cf_item_ids[order[:k_eff]].astype(int).tolist()

TOP_K_CF = 50
cf_lookup = {}
t0 = time.time()
print(f'\nĐang pre-compute CF lookup mặc định (k={TOP_K_CF}, chưa truyền city_name)...')
for i, uid in enumerate(cf_user_ids, 1):
    cf_lookup[int(uid)] = get_top_k_items_for_user_svd(int(uid), city_name=None, k=TOP_K_CF)
    if i % 10000 == 0 or i == len(cf_user_ids):
        print(f'  [{i}/{len(cf_user_ids)}] elapsed={time.time()-t0:.1f}s', flush=True)

np.save(OUT_DIR / 'cf_user_factors.npy', P)
np.save(OUT_DIR / 'cf_item_factors.npy', Q)
pd.DataFrame({'UserID': cf_user_ids}).to_csv(OUT_DIR / 'cf_user_ids.csv', index=False, encoding='utf-8-sig')
pd.DataFrame({'ResID':  cf_item_ids}).to_csv(OUT_DIR / 'cf_item_ids.csv', index=False, encoding='utf-8-sig')
with open(OUT_DIR / 'cf_lookup.pkl', 'wb') as f:
    pickle.dump(cf_lookup, f)

print(f'\nCF lookup mặc định: {len(cf_lookup)} users, top-{TOP_K_CF} mỗi user -> cf_lookup.pkl')

## 5. Hybrid — Union(CB, CF)

**Đầu vào**: `(user_id, current_item_id)`.

**Quy trình**:
1. Tra cứu song song:
   - `cb_list = cb_lookup[current_item_id]`  (đã trong cùng city)
   - `cf_list = cf_lookup[user_id]`          (toàn cục)
2. Filter `cf_list` về cùng thành phố với `current_item_id`.
3. **Union** với priority: items xuất hiện trong **CẢ HAI** list → CB-only → CF-only, dedupe, cắt top-K.

**Đầu ra**: DataFrame xếp hạng kèm cột `Source ∈ {BOTH, CB, CF}`.

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    if any(pd.isna(v) for v in [lat1, lon1, lat2, lon2]):
        return np.nan
    r = 6371.0
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return float(2 * r * np.arcsin(np.sqrt(a)))

def _distance_score(distance_km, decay_km=5.0):
    if pd.isna(distance_km):
        return 0.0
    return float(np.exp(-max(distance_km, 0.0) / decay_km))

def _rank_score(rank, total):
    if total <= 0:
        return 0.0
    return float(1.0 - ((rank - 1) / max(total, 1)))

def recommend_hybrid(user_id: int, current_item_id: int, city_name=None, k: int = 10,
                     cb_k: int = 50, cf_k: int = 50,
                     model_weight: float = 0.75, distance_weight: float = 0.25,
                     distance_decay_km: float = 5.0) -> pd.DataFrame:
    """Hybrid Union(CB, CF) + trọng số khoảng cách.

    Input online hiện tại:
      - user_id: user cần gợi ý
      - current_item_id: địa điểm user đang xem
      - city_name: tên thành phố lấy từ cột City trong places_lookup.csv

    Sau khi union CB + CF, mô hình tính thêm distance score:
      - địa điểm càng gần current_item_id thì DistanceScore càng cao
      - FinalScore = model_weight * ModelScore + distance_weight * DistanceScore
    """
    user_id = int(user_id)
    current_item_id = int(current_item_id)

    target_city = _normalize_city_name(city_name if city_name is not None else res2city.get(current_item_id))
    if not target_city:
        return pd.DataFrame()

    cb_list = [int(r) for r in get_top_k_items_by_item(current_item_id, city_name=target_city, k=cb_k)
               if int(r) != current_item_id]
    cf_list = [int(r) for r in get_top_k_items_for_user_svd(user_id, city_name=target_city, k=cf_k)
               if int(r) != current_item_id]

    cb_set, cf_set = set(cb_list), set(cf_list)
    union_ids = list(dict.fromkeys(cb_list + cf_list))
    if not union_ids:
        return pd.DataFrame()

    cb_rank = {rid: rank for rank, rid in enumerate(cb_list, 1)}
    cf_rank = {rid: rank for rank, rid in enumerate(cf_list, 1)}
    current_lat = res2lat.get(current_item_id)
    current_lon = res2lon.get(current_item_id)

    rows = []
    for rid in union_ids:
        source = 'BOTH' if rid in cb_set and rid in cf_set else ('CB' if rid in cb_set else 'CF')
        cb_score = _rank_score(cb_rank[rid], len(cb_list)) if rid in cb_rank else 0.0
        cf_score = _rank_score(cf_rank[rid], len(cf_list)) if rid in cf_rank else 0.0
        model_score = max(cb_score, cf_score)
        if source == 'BOTH':
            model_score = min(1.0, (cb_score + cf_score) / 2 + 0.10)

        rec_lat = res2lat.get(rid)
        rec_lon = res2lon.get(rid)
        distance_km = haversine_km(current_lat, current_lon, rec_lat, rec_lon)
        distance_score = _distance_score(distance_km, decay_km=distance_decay_km)
        final_score = model_weight * model_score + distance_weight * distance_score

        rows.append({
            'ResID': rid,
            'City': target_city,
            'Source': source,
            'ModelScore': round(model_score, 6),
            'DistanceKm': round(distance_km, 3) if not pd.isna(distance_km) else np.nan,
            'DistanceScore': round(distance_score, 6),
            'FinalScore': round(final_score, 6),
        })

    df = pd.DataFrame(rows).sort_values(['FinalScore', 'ModelScore', 'DistanceScore'], ascending=False).head(k)
    df['Rank'] = np.arange(1, len(df) + 1)
    df = df.merge(
        places_meta[['ResID', 'PlaceName', 'City', 'Latitude', 'Longitude', 'CategoryGroup', 'CategorySub']],
        on=['ResID', 'City'],
        how='left',
    )
    return df[['ResID', 'PlaceName', 'City', 'Latitude', 'Longitude',
               'CategoryGroup', 'CategorySub', 'DistanceKm', 'ModelScore',
               'DistanceScore', 'FinalScore', 'Rank', 'Source']]

print('Đã định nghĩa recommend_hybrid(user_id, current_item_id, city_name, ...) với Union(CB, CF) và trọng số khoảng cách.')

## 6. Test

Chọn vài user có nhiều ratings, dùng item đầu tiên họ đã rate làm `current_item_id`, gọi `recommend_hybrid()` và in ra kết quả.

In [ ]:
user_counts = np.diff(R_csr.indptr)
top_users = np.argsort(-user_counts)[:5]

for u_idx in top_users[:3]:
    uid = int(cf_user_ids[u_idx])
    rated = R_csr[u_idx].indices
    if len(rated) == 0:
        continue
    current_item = int(cf_item_ids[rated[0]])
    current_city = _normalize_city_name(res2city.get(current_item))
    src = places_meta[places_meta['ResID'] == current_item]
    if src.empty or not current_city:
        continue
    src_row = src.iloc[0]
    print(f"\n=== User={uid} | Item đang xem={current_item} "
          f"({src_row['PlaceName']} - {current_city}) ===")
    df = recommend_hybrid(uid, current_item, city_name=current_city, k=10)
    if df.empty:
        print('(Không có gợi ý phù hợp)')
    else:
        print(df[['PlaceName', 'City', 'DistanceKm', 'ModelScore', 'DistanceScore',
                  'FinalScore', 'Rank', 'Source']].to_string(index=False))

## 7. Đánh giá mô hình - RMSE

Đánh giá rating prediction trên thang 5 sao:

- Collaborative Filtering: dùng Funk SVD từ thư viện Surprise.
- Content-Based: mô phỏng dự đoán rating bằng item-based kNN trên embedding, chỉ dùng `R_train_csr` để tránh data leakage.
- Hybrid RMSE được tính thêm để tham khảo, còn Hybrid Union ở bước 5 vẫn là flow gợi ý Top-K chính.

In [ ]:
def predict_content_based_rating(uid, iid, R_matrix, global_mean_rating):
    start, end = R_matrix.indptr[int(uid)], R_matrix.indptr[int(uid) + 1]
    user_rated_items = R_matrix.indices[start:end]
    user_ratings = R_matrix.data[start:end]

    target_resid = int(cf_item_ids[int(iid)])
    target_emb_idx = res2content_idx.get(target_resid)
    if target_emb_idx is None or len(user_rated_items) == 0:
        return float(global_mean_rating)

    rated_emb_indices = []
    rated_values = []
    for item_idx, rating_value in zip(user_rated_items, user_ratings):
        rated_resid = int(cf_item_ids[int(item_idx)])
        emb_idx = res2content_idx.get(rated_resid)
        if emb_idx is not None:
            rated_emb_indices.append(emb_idx)
            rated_values.append(float(rating_value))

    if not rated_emb_indices:
        return float(global_mean_rating)

    sims = embeddings[rated_emb_indices] @ embeddings[target_emb_idx]
    positive_mask = sims > 0
    if not np.any(positive_mask):
        pred = np.mean(rated_values)
    else:
        sims_pos = sims[positive_mask]
        ratings_pos = np.array(rated_values, dtype=np.float32)[positive_mask]
        pred = float(np.dot(sims_pos, ratings_pos) / np.sum(sims_pos))

    return float(np.clip(pred, 0.5, 5.0))

print('Đang đánh giá CF (Funk SVD)...')
train_predictions_cf = cf_algo.test(train_set.build_testset())
val_predictions_cf = cf_algo.test(val_set)
test_predictions_cf = cf_algo.test(test_set)

cf_train_rmse = accuracy.rmse(train_predictions_cf, verbose=False)
cf_val_rmse = accuracy.rmse(val_predictions_cf, verbose=False)
cf_test_rmse = accuracy.rmse(test_predictions_cf, verbose=False)

print('Đang đánh giá Content-Based kNN...')
true_ratings_val = [true_r for uid, iid, true_r in val_set]
cb_preds_val = [predict_content_based_rating(uid, iid, R_train_csr, global_mean_5star) for uid, iid, _ in val_set]
cb_val_rmse = math.sqrt(mean_squared_error(true_ratings_val, cb_preds_val))

true_ratings_test = [true_r for uid, iid, true_r in test_set]
cb_preds_test = [predict_content_based_rating(uid, iid, R_train_csr, global_mean_5star) for uid, iid, _ in test_set]
cb_test_rmse = math.sqrt(mean_squared_error(true_ratings_test, cb_preds_test))

print('Đang tối ưu alpha Hybrid RMSE trên validation set...')
cf_preds_val = [pred.est for pred in val_predictions_cf]
cf_preds_test = [pred.est for pred in test_predictions_cf]

alphas = np.arange(0.0, 1.01, 0.05)
best_alpha = 0.0
best_hybrid_val_rmse = float('inf')
for alpha in alphas:
    hybrid_val_preds = [alpha * cf + (1 - alpha) * cb for cf, cb in zip(cf_preds_val, cb_preds_val)]
    hybrid_val_rmse = math.sqrt(mean_squared_error(true_ratings_val, hybrid_val_preds))
    if hybrid_val_rmse < best_hybrid_val_rmse:
        best_hybrid_val_rmse = hybrid_val_rmse
        best_alpha = float(alpha)

hybrid_test_preds = [best_alpha * cf + (1 - best_alpha) * cb for cf, cb in zip(cf_preds_test, cb_preds_test)]
hybrid_test_rmse = math.sqrt(mean_squared_error(true_ratings_test, hybrid_test_preds))

rmse_df = pd.DataFrame([
    {'Model': 'Collaborative Filtering (Funk SVD)', 'Split': 'Train', 'RMSE': cf_train_rmse},
    {'Model': 'Collaborative Filtering (Funk SVD)', 'Split': 'Validation', 'RMSE': cf_val_rmse},
    {'Model': 'Collaborative Filtering (Funk SVD)', 'Split': 'Test', 'RMSE': cf_test_rmse},
    {'Model': 'Content-Based kNN', 'Split': 'Validation', 'RMSE': cb_val_rmse},
    {'Model': 'Content-Based kNN', 'Split': 'Test', 'RMSE': cb_test_rmse},
    {'Model': f'Hybrid Weighted (alpha={best_alpha:.2f})', 'Split': 'Validation', 'RMSE': best_hybrid_val_rmse},
    {'Model': f'Hybrid Weighted (alpha={best_alpha:.2f})', 'Split': 'Test', 'RMSE': hybrid_test_rmse},
])

rmse_df.to_csv(OUT_DIR / 'rmse_results_5star.csv', index=False, encoding='utf-8-sig')
print('\n=== KẾT QUẢ RMSE THANG 5 SAO ===')
print(rmse_df.to_string(index=False))
print('\nBộ siêu tham số tốt nhất:', best_svd_params)
print(f'Đã lưu: {OUT_DIR / "rmse_results_5star.csv"}')

### 7.1 So sánh rating dự đoán và rating thực tế, kiểm tra leakage/overfitting

Cell này in mẫu rating thực tế và rating dự đoán cho một user cụ thể, sau đó đưa ra chẩn đoán nhanh về overfitting và data leakage.

In [ ]:
from collections import Counter

USER_ID_TO_INSPECT = None  # Đổi thành UserID cụ thể, ví dụ: 12345. Để None sẽ tự chọn user có nhiều rating trong test set.
N_PREDICTION_EXAMPLES = 10


def pick_user_from_split(split_set, user_id=None):
    if user_id is not None:
        target = int(user_id)
        matched = [(uid, iid, true_r) for uid, iid, true_r in split_set if int(cf_user_ids[int(uid)]) == target]
        if matched:
            return target, matched
        print(f'Không tìm thấy UserID={target} trong split đang inspect; tự động chọn user khác.')

    counts = Counter(int(uid) for uid, _, _ in split_set)
    selected_uid_idx, _ = counts.most_common(1)[0]
    selected_user_id = int(cf_user_ids[selected_uid_idx])
    matched = [(uid, iid, true_r) for uid, iid, true_r in split_set if int(uid) == selected_uid_idx]
    return selected_user_id, matched


def build_prediction_inspection(split_set, user_id=None, n=10):
    selected_user_id, samples = pick_user_from_split(split_set, user_id=user_id)
    rows = []
    for uid, iid, true_r in samples[:n]:
        uid_idx = int(uid)
        iid_idx = int(iid)
        resid = int(cf_item_ids[iid_idx])
        meta = places_meta.loc[places_meta['ResID'].astype(int) == resid]
        if meta.empty:
            place_name, city = '', ''
        else:
            place_name = meta.iloc[0]['PlaceName']
            city = meta.iloc[0]['City']

        cf_pred = float(np.clip(cf_algo.predict(uid, iid).est, 0.5, 5.0))
        cb_pred = float(predict_content_based_rating(uid_idx, iid_idx, R_train_csr, global_mean_5star))
        hybrid_pred = float(np.clip(best_alpha * cf_pred + (1 - best_alpha) * cb_pred, 0.5, 5.0))

        rows.append({
            'UserID': int(cf_user_ids[uid_idx]),
            'ResID': resid,
            'PlaceName': place_name,
            'City': city,
            'Actual_5Star': round(float(true_r), 3),
            'CF_Pred_5Star': round(cf_pred, 3),
            'CB_Pred_5Star': round(cb_pred, 3),
            'Hybrid_Pred_5Star': round(hybrid_pred, 3),
            'CF_Abs_Error': round(abs(cf_pred - float(true_r)), 3),
            'CB_Abs_Error': round(abs(cb_pred - float(true_r)), 3),
            'Hybrid_Abs_Error': round(abs(hybrid_pred - float(true_r)), 3),
        })

    return selected_user_id, pd.DataFrame(rows)


inspect_user_id, prediction_examples_df = build_prediction_inspection(
    test_set,
    user_id=USER_ID_TO_INSPECT,
    n=N_PREDICTION_EXAMPLES,
)

prediction_examples_df.to_csv(OUT_DIR / 'prediction_examples_5star.csv', index=False, encoding='utf-8-sig')
print(f'Một vài rating dự đoán và rating thực tế của UserID={inspect_user_id} trên TEST split')
display(prediction_examples_df)

cf_train_val_gap = cf_val_rmse - cf_train_rmse
cf_train_test_gap = cf_test_rmse - cf_train_rmse
cf_val_test_gap = abs(cf_test_rmse - cf_val_rmse)
cb_val_test_gap = abs(cb_test_rmse - cb_val_rmse)
hybrid_val_test_gap = abs(hybrid_test_rmse - best_hybrid_val_rmse)

if cf_train_test_gap > 0.30:
    overfit_status = 'Cảnh báo: CF có dấu hiệu overfitting rõ, train RMSE thấp hơn test khá nhiều.'
elif cf_train_test_gap > 0.15:
    overfit_status = 'Theo dõi: CF có khoảng cách train-test vừa phải, cần kiểm tra thêm theo từng nhóm user/item.'
else:
    overfit_status = 'Ổn: chưa thấy dấu hiệu overfitting mạnh qua RMSE train/validation/test.'

if max(cf_val_test_gap, cb_val_test_gap, hybrid_val_test_gap) > 0.15:
    generalization_status = 'Validation và test lệch nhau tương đối lớn; nên kiểm tra lại cách split hoặc độ phân bố data.'
else:
    generalization_status = 'Validation và test khá gần nhau; khả năng tổng quát hóa ổn định hơn.'

leakage_notes = [
    'CF: train bằng train_set, RMSE tính trên validation/test riêng nên không thấy leakage trực tiếp trong rating prediction.',
    'CB: predict rating chỉ dùng R_train_csr làm lịch sử user, tránh dùng rating validation/test làm đầu vào.',
    'Metadata/content embeddings dùng thông tin item, không dùng rating mục tiêu; đây không phải rating leakage.',
    'Lưu ý: recommend_hybrid/CF lookup đang mask item đã tương tác bằng R_csr đầy đủ. Nếu dùng để đánh giá ranking holdout, nên mask bằng R_train_csr để tránh nhìn thấy lịch sử validation/test.',
]

diagnosis_df = pd.DataFrame([
    {'Check': 'CF train RMSE', 'Value': round(cf_train_rmse, 4), 'Interpretation': 'Baseline fit trên train'},
    {'Check': 'CF validation RMSE', 'Value': round(cf_val_rmse, 4), 'Interpretation': 'Sai số trên validation'},
    {'Check': 'CF test RMSE', 'Value': round(cf_test_rmse, 4), 'Interpretation': 'Sai số trên test'},
    {'Check': 'CF train-test gap', 'Value': round(cf_train_test_gap, 4), 'Interpretation': overfit_status},
    {'Check': 'CF val-test gap', 'Value': round(cf_val_test_gap, 4), 'Interpretation': generalization_status},
    {'Check': 'CB val-test gap', 'Value': round(cb_val_test_gap, 4), 'Interpretation': generalization_status},
    {'Check': 'Hybrid val-test gap', 'Value': round(hybrid_val_test_gap, 4), 'Interpretation': generalization_status},
])

diagnosis_df.to_csv(OUT_DIR / 'overfitting_leakage_diagnosis.csv', index=False, encoding='utf-8-sig')
print('\n=== CHẨN ĐOÁN OVERFITTING / DATA LEAKAGE ===')
display(diagnosis_df)
print('\nGhi chú về leakage:')
for note in leakage_notes:
    print('-', note)
print(f'\nĐã lưu: {OUT_DIR / "prediction_examples_5star.csv"}')
print(f'Đã lưu: {OUT_DIR / "overfitting_leakage_diagnosis.csv"}')

## 8. Trích xuất dữ liệu mẫu ra CSV

In [ ]:
SAMPLE_DIR = OUT_DIR / 'samples_csv'
SAMPLE_DIR.mkdir(parents=True, exist_ok=True)

print('Đang trích xuất dữ liệu mẫu ra CSV...')

places_meta.head(100).to_csv(SAMPLE_DIR / 'sample_places_meta.csv', index=False, encoding='utf-8-sig')
places_meta[['ResID', 'PlaceName', 'City', 'Latitude', 'Longitude']].head(100).to_csv(SAMPLE_DIR / 'sample_places_coordinates.csv', index=False, encoding='utf-8-sig')

R_coo = R.tocoo()
cf_input_df = pd.DataFrame({
    'UserID': cf_user_ids[R_coo.row[:100]],
    'ResID': cf_item_ids[R_coo.col[:100]],
    'Rating_5Star': R_coo.data[:100],
})
cf_input_df.to_csv(SAMPLE_DIR / 'sample_cf_input_ratings_5star.csv', index=False, encoding='utf-8-sig')

cb_sample = {k: cb_lookup[k] for k in list(cb_lookup.keys())[:100]}
cb_df = pd.DataFrame([{'ResID': k, 'Similar_Items': v} for k, v in cb_sample.items()])
cb_df.to_csv(SAMPLE_DIR / 'sample_cb_lookup.csv', index=False, encoding='utf-8-sig')

cf_sample = {k: cf_lookup.get(k, []) for k in list(cf_lookup.keys())[:100]}
cf_df = pd.DataFrame([{'UserID': k, 'Recommended_Items_Top50_ByCityName': v} for k, v in cf_sample.items()])
cf_df.to_csv(SAMPLE_DIR / 'sample_cf_lookup.csv', index=False, encoding='utf-8-sig')

pd.DataFrame(P[:100]).to_csv(SAMPLE_DIR / 'sample_user_factors_P.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(Q[:100]).to_csv(SAMPLE_DIR / 'sample_item_factors_Q.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(embeddings[:100]).to_csv(SAMPLE_DIR / 'sample_content_embeddings.csv', index=False, encoding='utf-8-sig')

if 'rmse_df' in globals():
    rmse_df.to_csv(SAMPLE_DIR / 'sample_rmse_results_5star.csv', index=False, encoding='utf-8-sig')
if 'prediction_examples_df' in globals():
    prediction_examples_df.to_csv(SAMPLE_DIR / 'sample_prediction_examples_5star.csv', index=False, encoding='utf-8-sig')
if 'diagnosis_df' in globals():
    diagnosis_df.to_csv(SAMPLE_DIR / 'sample_overfitting_leakage_diagnosis.csv', index=False, encoding='utf-8-sig')

print(f'Đã lưu thành công các file CSV mẫu tại: {SAMPLE_DIR}')
display(cf_input_df.head())